In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd, json, glob

ROOT = Path('/content/drive/MyDrive/US_ETF')
CACHE = ROOT/'directional_research/open_revalidation_1m_alpaca_v1/iex'
RES = ROOT/'model_lab_v1/results/open_revalidation_v1'

print('='*96)
print('KALMAN OPEN REVALIDATION — POST ALPACA BACKFILL CHECK')
print('='*96)
print('cache_exists =', CACHE.exists())
files = [p for p in CACHE.rglob('*') if p.is_file()] if CACHE.exists() else []
print('cache_files =', len(files))

status = RES/'alpaca_backfill_status_latest.csv'
if status.exists():
    s=pd.read_csv(status)
    print('\n[BACKFILL STATUS] rows=',len(s),' columns=',list(s.columns))
    print(s.tail(5).to_string(index=False))
else: print('\n[WARN] missing',status)

# Find the newest audit-like table produced by the research workflow.
cands=[]
for pat in ['*audit*.parquet','*revalidation*.parquet','*audit*.csv','*revalidation*.csv']:
    cands += list(RES.rglob(pat)) if RES.exists() else []
cands=sorted(set(cands), key=lambda p:p.stat().st_mtime, reverse=True)
print('\n[AUDIT CANDIDATES]')
for p in cands[:20]: print(p)

df=None; chosen=None
for p in cands:
    try:
        x=pd.read_parquet(p) if p.suffix=='.parquet' else pd.read_csv(p)
        if len(x) and any(c in x.columns for c in ['revalidation_data_ready','entry_price_iex','open_0_price_iex']):
            df=x; chosen=p; break
    except Exception: pass

if df is None:
    print('\n[FAIL] No regenerated Open revalidation audit found.')
    print('Backfill is present, but the audit builder must be rerun before promotion gates.')
else:
    print('\n[OPEN REVALIDATION] source=',chosen)
    print('total_rows =',len(df))
    if 'revalidation_data_ready' in df:
        ready=df['revalidation_data_ready'].fillna(False).astype(bool)
        print('ready_rows =',int(ready.sum()))
        print('not_ready_rows =',int((~ready).sum()))
        print('ready_rate =',f'{ready.mean():.2%}')
    fields=['entry_price_iex','fixed4_exit_price_iex','prev_close_price_iex','open_0_price_iex','open_5_price_iex','open_15_price_iex']
    print('\n[MISSING PRICE FIELDS]')
    for c in fields:
        print(f'{c} =', int(df[c].isna().sum()) if c in df else 'COLUMN_MISSING')

print('\n[GATE] R9 promotion remains BLOCKED until open coverage + prospective + fold + bootstrap gates pass.')
